based on the data we can build content based recommender system only

In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
df = pd.read_csv('../datasets/appartments.csv').drop(22)
df.head()


,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."


In [3]:
df.shape


(246, 7)

In [4]:
df.iloc[2].NearbyLocations


"['AIPL Business Club Sector 62', 'Heritage Xperiential Learning School', 'CK Birla Hospital', 'Paras Trinity Mall Sector 63', 'Rapid Metro Station Sector 56']"

In [5]:
df.iloc[2].LocationAdvantages


"{'AIPL Business Club Sector 62': '2.7 Km', 'Heritage Xperiential Learning School': '2 Km', 'CK Birla Hospital': '2.5 Km', 'Paras Trinity Mall Sector 63': '3.5 Km', 'Rapid Metro Station Sector 56': '3.8 Km', 'De Adventure Park': '6.8 Km', 'Golf Course Ext Rd': '99 Meter', 'DoubleTree by Hilton Hotel Gurgaon': '3.6 Km', 'KIIT College of Engineering Sohna Road': '8.4 Km', 'Mehrauli-Gurgaon Road': '11.8 Km', 'Indira Gandhi International Airport': '21.1 Km', 'Nirvana Rd': '160 Meter', 'TERI Golf Course': '8.7 Km'}"

we can see that 'NearbyLocations' is subset of 'LocationAdvantages'. We can ignore 'NearbyLocations' since the info in it can be gained through 'LocationAdvantages'.

In [6]:

df.iloc[2].PriceDetails


"{'3 BHK': {'building_type': 'Independent Floor', 'area_type': 'Super Built-up Area', 'area': '1,800 - 3,150 sq.ft.', 'price-range': '₹ 2.43 - 15.75 Cr'}, '4 BHK': {'building_type': 'Independent Floor', 'area_type': 'Super Built-up Area', 'area': '2,750 - 4,500 sq.ft.', 'price-range': '₹ 3.36 - 22.5 Cr'}, 'Land': {'building_type': '', 'area_type': 'Plot Area', 'area': '500 - 4,329 sq.ft.', 'price-range': '₹ 2.05 - 41.13 Cr'}}"

In [7]:
df.iloc[2].PropertySubName


'Land, 3, 4 BHK Independent Floor in Sector 63, Gurgaon'

if we ignore the sector information then rest of the info of 'PropertySubName' is present in 'PriceDetails'

In [8]:
df.iloc[2].TopFacilities


"['Terrace Garden', 'Gazebo', 'Fountain', 'Amphitheatre', 'Party Lawn', 'Basketball Court', 'Badminton Court', 'Yoga/Meditation Area', 'Indoor Games']"

In [9]:
df[['PropertyName', 'TopFacilities']].sample(10)


,PropertyName,TopFacilities
86,DLF The Ultima,"['Mini Theatre', 'Swimming Pool', 'Theater Hom..."
36,Emaar Gurgaon Greens,"['Swimming Pool', 'Flower Garden', 'Golf Cours..."
94,Imperia The Esfera,"['Food Court', 'Swimming Pool', 'Reading Loung..."
71,International City by SOBHA Phase 2,"['Swimming Pool', 'Theater Home', 'School', 'S..."
201,Vipul Belmonte,"['Swimming Pool', 'ATM', 'Water Softener Plant..."
101,Suncity Avenue 76,"['School', 'High Speed Elevators', 'Creche/Day..."
234,Unitech Escape,"['Swimming Pool', 'Business Lounge', 'Jacuzzi'..."
57,DLF The Magnolias,"['Mini Theatre', 'Swimming Pool', 'Football', ..."
245,BPTP Spacio,"['Swimming Pool', 'Card Room', 'Piped Gas', 'P..."
54,Birla Navya Avik,"['Football', 'Skating Rink', 'Cricket Pitch', ..."


### TopFacilities Recommendation System

In [10]:
df['TopFacilities'][0]


"['Swimming Pool', 'Salon', 'Restaurant', 'Spa', 'Cafeteria', 'Sun Deck', '24x7 Security', 'Club House', 'Gated Community']"

In [11]:
# records in the TopFacilities column are strings that contain lists
# extractings the lists from the string
def extract_list(s):
    return re.findall(r"'(.*?)'", s)

df['FacilitiesString'] = df['TopFacilities'].apply(extract_list)


In [12]:
df['FacilitiesString'][0]


['Swimming Pool',
 'Salon',
 'Restaurant',
 'Spa',
 'Cafeteria',
 'Sun Deck',
 '24x7 Security',
 'Club House',
 'Gated Community']

In [13]:
df.head()


,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,FacilitiesString
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'...","[Swimming Pool, Salon, Restaurant, Spa, Cafete..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G...","[Bowling Alley, Mini Theatre, Manicured Garden..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph...","[Terrace Garden, Gazebo, Fountain, Amphitheatr..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi...","[Swimming Pool, Volley Ball Court, Aerobics Ce..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ...","[Mini Theatre, Doctor on Call, Concierge Servi..."


In [14]:
# concatenating list items into a single string
df['FacilitiesString'] = df['FacilitiesString'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')


In [15]:
df['FacilitiesString'][0]


'Swimming Pool Salon Restaurant Spa Cafeteria Sun Deck 24x7 Security Club House Gated Community'

In [16]:
# vectorizing the FacilitiesString column
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df['FacilitiesString'])


In [17]:
tfidf_matrix.toarray().shape


(246, 953)

now facilitiesString is converted to 953 dimensional vector.

In [18]:
# using cosine similarity to find similar apartments
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix) # finds similarity scores between all possible pairs of apartments
cosine_sim.shape


(246, 246)

In [19]:
def recommender_using_facilities(property_name, cosine_sim=cosine_sim):
    # get the index of the property
    idx = df.index[df['PropertyName'] == property_name].tolist()[0]
    
    # get the pairwise similarity scores of all properties with that property
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # sort the properties based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # get the scores of the 10 most similar properties
    sim_scores = sim_scores[1:11]
    
    # get the property indices
    property_indices = [i[0] for i in sim_scores]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices],
        'similarity_score': sim_scores
    })
    # return the top 10 most similar properties
    return recommendations_df


In [20]:
recommender_using_facilities('DLF The Arbour')


,PropertyName,similarity_score
64,Ace Palm Floors,"(63, 0.4529382062441955)"
217,Yashika 104,"(216, 0.4199606322926784)"
93,JMS The Nation,"(92, 0.4166584649363288)"
154,India Rashtra,"(153, 0.398954234680194)"
0,Smartworld One DXP,"(0, 0.38885046199432893)"
73,BPTP Green Oaks,"(72, 0.38821619878613484)"
63,Vatika Aspiration,"(62, 0.3767977219141552)"
18,Whiteland Blissville,"(18, 0.35741312385828006)"
186,Anant Raj Ashok Estate,"(185, 0.35334766728957717)"
243,Pyramid Urban Homes 2,"(242, 0.34844371350826064)"


### PriceDetails Recommendation System

In [21]:
df[['PropertyName', 'PriceDetails']].sample(10)


,PropertyName,PriceDetails
20,Tulip Monsella,"{'3 BHK': {'building_type': 'Apartment', 'area..."
37,Oxirich Chintamanis,"{'3 BHK': {'building_type': 'Apartment', 'area..."
239,La Lagune,"{'3 BHK': {'building_type': 'Apartment', 'area..."
33,Godrej Nature Plus Serenity,"{'2 BHK': {'building_type': 'Apartment', 'area..."
81,AIPL The Peaceful Homes,"{'2 BHK': {'building_type': 'Apartment', 'area..."
179,Landmark The Homes 81,"{'1 BHK': {'building_type': 'Apartment', 'area..."
124,Shree Vardhman Flora,"{'2 BHK': {'building_type': 'Apartment', 'area..."
225,Signature Global Prime,"{'2 BHK': {'building_type': 'Apartment', 'area..."
154,India Rashtra,"{'Land': {'building_type': '', 'area_type': 'P..."
138,Godrej Oasis,"{'2 BHK': {'building_type': 'Apartment', 'area..."


In [22]:
import pandas as pd
import json

# Load the dataset
df_appartments = pd.read_csv('../datasets/appartments.csv').drop(22)

# Function to parse and extract the required features from the PriceDetails column
def refined_parse_modified_v2(detail_str):
    try:
        details = json.loads(detail_str.replace("'", "\""))
    except:
        return {}

    extracted = {}
    for bhk, detail in details.items():
        # Extract building type
        extracted[f'building type_{bhk}'] = detail.get('building_type')

        # Parsing area details
        area = detail.get('area', '')
        area_parts = area.split('-')
        if len(area_parts) == 1:
            try:
                value = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area low {bhk}'] = value
                extracted[f'area high {bhk}'] = value
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None
        elif len(area_parts) == 2:
            try:
                extracted[f'area low {bhk}'] = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area high {bhk}'] = float(area_parts[1].replace(',', '').replace(' sq.ft.', '').strip())
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None

        # Parsing price details
        price_range = detail.get('price-range', '')
        price_parts = price_range.split('-')
        if len(price_parts) == 2:
            try:
                extracted[f'price low {bhk}'] = float(price_parts[0].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                extracted[f'price high {bhk}'] = float(price_parts[1].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                if 'L' in price_parts[0]:
                    extracted[f'price low {bhk}'] /= 100
                if 'L' in price_parts[1]:
                    extracted[f'price high {bhk}'] /= 100
            except:
                extracted[f'price low {bhk}'] = None
                extracted[f'price high {bhk}'] = None

    return extracted
# Apply the refined parsing and generate the new DataFrame structure
data_refined = []

for _, row in df_appartments.iterrows():
    features = refined_parse_modified_v2(row['PriceDetails'])
    
    # Construct a new row for the transformed dataframe
    new_row = {'PropertyName': row['PropertyName']}
    
    # Populate the new row with extracted features
    for config in ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']:
        new_row[f'building type_{config}'] = features.get(f'building type_{config}')
        new_row[f'area low {config}'] = features.get(f'area low {config}')
        new_row[f'area high {config}'] = features.get(f'area high {config}')
        new_row[f'price low {config}'] = features.get(f'price low {config}')
        new_row[f'price high {config}'] = features.get(f'price high {config}')
    
    data_refined.append(new_row)

df_final_refined = pd.DataFrame(data_refined).set_index('PropertyName')


In [23]:
df_final_refined.head()


,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,...,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,None,NaN,NaN,NaN,NaN,Apartment,1370.0,1370.0,2.0000,2.40,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
M3M Crown,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,,500.0,4329.0,2.05,41.13
Sobha City,None,NaN,NaN,NaN,NaN,Apartment,1381.0,1692.0,1.5500,3.21,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Signature Global City 93,None,NaN,NaN,NaN,NaN,Independent Floor,981.0,1118.0,0.9301,1.06,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN


In [24]:
df_final_refined['building type_Land'] = df_final_refined['building type_Land'].replace({'':'Land'})
df_final_refined.head()


,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,...,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,None,NaN,NaN,NaN,NaN,Apartment,1370.0,1370.0,2.0000,2.40,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
M3M Crown,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,Land,500.0,4329.0,2.05,41.13
Sobha City,None,NaN,NaN,NaN,NaN,Apartment,1381.0,1692.0,1.5500,3.21,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Signature Global City 93,None,NaN,NaN,NaN,NaN,Independent Floor,981.0,1118.0,0.9301,1.06,...,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN


40 new cols are made. but the complex structure is converted into a simplified manner

In [25]:
df_final_refined.columns


Index(['building type_1 BHK', 'area low 1 BHK', 'area high 1 BHK',
       'price low 1 BHK', 'price high 1 BHK', 'building type_2 BHK',
       'area low 2 BHK', 'area high 2 BHK', 'price low 2 BHK',
       'price high 2 BHK', 'building type_3 BHK', 'area low 3 BHK',
       'area high 3 BHK', 'price low 3 BHK', 'price high 3 BHK',
       'building type_4 BHK', 'area low 4 BHK', 'area high 4 BHK',
       'price low 4 BHK', 'price high 4 BHK', 'building type_5 BHK',
       'area low 5 BHK', 'area high 5 BHK', 'price low 5 BHK',
       'price high 5 BHK', 'building type_6 BHK', 'area low 6 BHK',
       'area high 6 BHK', 'price low 6 BHK', 'price high 6 BHK',
       'building type_1 RK', 'area low 1 RK', 'area high 1 RK',
       'price low 1 RK', 'price high 1 RK', 'building type_Land',
       'area low Land', 'area high Land', 'price low Land', 'price high Land'],
      dtype='object')

In [26]:
categorical_columns = df_final_refined.select_dtypes(include=['object']).columns.tolist()


In [27]:
categorical_columns


['building type_1 BHK',
 'building type_2 BHK',
 'building type_3 BHK',
 'building type_4 BHK',
 'building type_5 BHK',
 'building type_6 BHK',
 'building type_1 RK',
 'building type_Land']

In [28]:
# performing OHE on all categorical columns
ohe_df = pd.get_dummies(df_final_refined, columns=categorical_columns, drop_first=True)
ohe_df.fillna(0, inplace=True)


In [29]:
ohe_df


,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.0,0.0,0.00,0.0000,1370.0,1370.0,2.0000,2.40,1850.0,2050.0,...,False,False,False,False,False,False,False,False,False,False
M3M Crown,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1605.0,2170.0,...,False,False,False,False,False,False,False,False,False,False
Adani Brahma Samsara Vilasa,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1800.0,3150.0,...,False,False,True,False,False,True,False,False,False,False
Sobha City,0.0,0.0,0.00,0.0000,1381.0,1692.0,1.5500,3.21,1711.0,2343.0,...,False,False,False,False,False,False,False,False,False,False
Signature Global City 93,0.0,0.0,0.00,0.0000,981.0,1118.0,0.9301,1.06,1235.0,1530.0,...,True,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.0,0.0,0.00,0.0000,964.0,964.0,0.0000,0.00,1127.0,1127.0,...,False,False,False,False,False,False,False,False,False,False
Pyramid Urban Homes 2,335.0,398.0,23.45,0.2786,500.0,625.0,0.0000,0.00,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
Satya The Hermitage,0.0,0.0,0.00,0.0000,1450.0,1450.0,0.0000,0.00,1991.0,1991.0,...,False,False,False,False,False,False,False,False,False,False


In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

ohe_df_scaled = pd.DataFrame(scaler.fit_transform(ohe_df), columns=ohe_df.columns, index=ohe_df.index)


In [31]:
ohe_df_scaled.head()


,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,-0.252266,-0.169584,-0.105197,-0.082332,1.223499,1.020101,-0.173712,1.158423,0.553787,0.370864,...,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
M3M Crown,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.293086,0.472749,...,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Adani Brahma Samsara Vilasa,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.500583,1.304803,...,-0.289310,-0.063888,2.683282,-0.063888,-0.171139,3.924283,-0.236208,-0.111111,-0.216353,-0.063888
Sobha City,-0.252266,-0.169584,-0.105197,-0.082332,1.240497,1.470610,-0.198425,1.680336,0.405879,0.619632,...,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Signature Global City 93,-0.252266,-0.169584,-0.105197,-0.082332,0.622383,0.667529,-0.232468,0.295011,-0.100626,-0.070634,...,3.456497,-0.063888,2.683282,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888


In [32]:
cosine_sim2 = cosine_similarity(ohe_df_scaled)


In [33]:
cosine_sim2.shape


(246, 246)

In [34]:
def price_details_recommendation_system(property_name, top_n = 10):
    # get similarity scores for the given property
    sim_scores = list(enumerate(cosine_sim2[ohe_df_scaled.index.get_loc(property_name)]))
    # sort the properties based on similarity scores
    sorted_sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # get the top n most similar properties
    top_indices = [i[0] for i in sorted_sim_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_sim_scores[1:top_n+1]]
    
    # retrive property names
    top_properties = ohe_df_scaled.index[top_indices].tolist()
    
    # creat dataframe
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'similarity_score': top_scores
    })
    
    return recommendations_df

price_details_recommendation_system('M3M Golf Hills')


,PropertyName,similarity_score
0,AIPL The Peaceful Homes,0.955462
1,Smartworld One DXP,0.954670
2,Unitech Escape,0.953092
3,M3M Capital,0.951156
4,BPTP Terra,0.943128
5,Sobha City,0.928748
6,Unitech Harmony,0.925164
7,Corona Optus,0.919231
8,Puri Emerald Bay,0.917345
9,Ireo Skyon,0.915991


### LocationAdvantages Based Recommendation System

In [35]:
df[['PropertyName', 'LocationAdvantages']]


,PropertyName,LocationAdvantages
0,Smartworld One DXP,"{'Bajghera Road': '800 Meter', 'Palam Vihar Ha..."
1,M3M Crown,"{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N..."
2,Adani Brahma Samsara Vilasa,"{'AIPL Business Club Sector 62': '2.7 Km', 'He..."
3,Sobha City,"{'The Shikshiyan School': '2.9 KM', 'WTC Plaza..."
4,Signature Global City 93,"{'Pranavananda Int. School': '450 m', 'DLF Sit..."
...,...,...
242,DLF Princeton Estate,"{'Sector 42-43 Metro Station': '1.8 Km', 'Para..."
243,Pyramid Urban Homes 2,{'Aarvy Healthcare Super Speciality': '1.8 KM'...
244,Satya The Hermitage,"{'Dwarka Expressway': '1.2 Km', 'S N Internati..."
245,BPTP Spacio,"{'Suncity School': '0.2 Km', 'Gurugram Road': ..."


In [36]:
pd.set_option('display.max_columns', None)


In [37]:
df['LocationAdvantages'][0]


"{'Bajghera Road': '800 Meter', 'Palam Vihar Halt': '2.5 KM', 'DPSG Palam Vihar': '3.1 KM', 'Park Hospital': '3.1 KM', 'Gurgaon Railway Station': '4.9 KM', 'The NorthCap University': '5.4 KM', 'Dwarka Expy': '1.2 KM', 'Hyatt Place Gurgaon Udyog Vihar': '7.7 KM', 'Dwarka Sector 21, Metro Station': '7.2 KM', 'Pacific D21 Mall': '7.4 KM', 'Indira Gandhi International Airport': '14.7 KM', 'Hamoni Golf Camp': '6.2 KM', 'Fun N Food Waterpark': '8.8 KM', 'Accenture DDC5': '9 KM'}"

In [38]:
def km_to_m(dist):
    try:
        if "Km" in dist or 'KM' in dist:
            return float(dist.split()[0]) * 1000
        elif 'Meter' in dist or 'meter' in dist:
            return float(dist.split()[0])
        else:
            return None
    except:
        return None


In [39]:
import ast

# extract distances
location_matrix = {}
for index, row in df.iterrows():
    distances = {}
    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        distances[location] = km_to_m(distance)
    location_matrix[index] = distances


In [40]:
location_df = pd.DataFrame.from_dict(location_matrix, orient='index')
location_df.index = df['PropertyName']
location_df.head()


,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Fun N Food Waterpark,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Dwarka Sector 21 Metro Station,Dwarka Expressway,Fun N Food Water Park,Tau DeviLal Sports Complex,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,Golf Course Ext Rd,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Rions Hospital,Gurgaon,Dwarka Sector 21,Nehru Stadium,Fun N Food WaterPark,IGI Airport,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,NH48,Garhi Harsaru Junction,Manesar Golf Course,AapnoGhar,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109","Euro International School, Sector- 109",Jai Sai Ram Hospital,Aryan Hospital,Idea Cosmic Plaza,Indira Gandhi Intl Airport,Royal Institute Of Science & Management,Pataudi Road,Holiday Inn Sector 90,RPS International School,Aarvy Healthcare Hospital,Iris Broadway Mall,Imperia Mindspace,AIPL Business Tower,Heritage School,"Lotus Valley Intl School, Gurgaon",Gurugram University,Sector 55-56 Metro Station,Omaxe Gurgaon Mall,Sushant University,"Badshahpur Sohna Rd Hwy,Sector 48",Naurangpur Cricket Stadium,Naurangpur Road,National Highway 48,Vatika Town Square-INXT,Ompee International School,Manesar Bus Stand,Yashlok Medical Centre,Euro International School,WorldMark Gurgaon,Capital Cyberscape,The Shriram Millennium School,DoubleTree by Hilton Hotel,Badshahpur Sohna Hwy,Nakhrola Stadium,Delhi - Jaipur Expressway,Vatika Town Square-INXT Mall,Savoy Suites,Bal Bharati Public School,Vatika Business Centre,Indira Gandhi Int. Airport,St. Xavier's High School,Miracles Apollo Cradle,Ambience Mall New,NH8,Hyatt Regency Gurgaon,Delhi Public School,Elan Miracle Mall,Miracles Apollo Cradle Spectra Hospital,Agri Business Management Collage,Delhi Jaipur Expressway,Grand Hyatt Gurgaon,Duke Horse Riding Club,PVR Drive In Cinema,W Pratiksha Hospital,Metro World Mall,Unicosmos School,Faridabad Gurgaon Road,Sohna Road,Bestech Business Tower,Appu Ghar,SkyJumper Trampoline Park,Axis Bank,KMP Expressway,Karma Lakelands,Jungle Safari & Trails,DPS Manesar,Medanta Hospital,Faridabad - Gurgaon Road,Lingaya's Lalita Devi Institute,ASF Insignia SEZ,Banjara Market Gurugram,Central Plaza Mall,"Paras Hospitals, Gurgaon",Badshahpur Sohna Rd Hwy,Vega School,Indian School of Hospitality,Vatika City Centre,Aatish Hospital,Info Technology Park Phase 2,Huda Metro Station,Southern Peripheral Rd,Global Ways School,Radisson Hotel,NH 248A,Sector 55/56 Metro Station,Mavens Inn,Sanar International Hospital,Sector 53-54 Metro Station,"IILM University, Gurugram",The Banyan Tree World School,The Big Tree Cafe,DLF Golf and Country Club,"Delhi Public School, Sector 84",Aarvy Hospital,DPG Degree College,Shivani public school,Baghera University,Kutumbh Hospital,Bijwasan Railway Station,Global Foyer Mall,Phase 2 Metro Station,Gurgaon Dreamz Mall,"Metro Hospital, Palam Vihar",Delhi Ajmer Expressway,Infinity Business Park,Huda metro station,Rion's Hospital,"Euro International School, Sector- 109.",Golf Course Ext Road,"Heritage Xperiential Learning, CRPF Rd",Sector 54 Chowk Metro Station

there are repeatation of landmarks with different annotations of same landmarks.

In [41]:
location_df.fillna(68000, inplace=True)
location_df.head()


,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Fun N Food Waterpark,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Dwarka Sector 21 Metro Station,Dwarka Expressway,Fun N Food Water Park,Tau DeviLal Sports Complex,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,Golf Course Ext Rd,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Rions Hospital,Gurgaon,Dwarka Sector 21,Nehru Stadium,Fun N Food WaterPark,IGI Airport,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,NH48,Garhi Harsaru Junction,Manesar Golf Course,AapnoGhar,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109","Euro International School, Sector- 109",Jai Sai Ram Hospital,Aryan Hospital,Idea Cosmic Plaza,Indira Gandhi Intl Airport,Royal Institute Of Science & Management,Pataudi Road,Holiday Inn Sector 90,RPS International School,Aarvy Healthcare Hospital,Iris Broadway Mall,Imperia Mindspace,AIPL Business Tower,Heritage School,"Lotus Valley Intl School, Gurgaon",Gurugram University,Sector 55-56 Metro Station,Omaxe Gurgaon Mall,Sushant University,"Badshahpur Sohna Rd Hwy,Sector 48",Naurangpur Cricket Stadium,Naurangpur Road,National Highway 48,Vatika Town Square-INXT,Ompee International School,Manesar Bus Stand,Yashlok Medical Centre,Euro International School,WorldMark Gurgaon,Capital Cyberscape,The Shriram Millennium School,DoubleTree by Hilton Hotel,Badshahpur Sohna Hwy,Nakhrola Stadium,Delhi - Jaipur Expressway,Vatika Town Square-INXT Mall,Savoy Suites,Bal Bharati Public School,Vatika Business Centre,Indira Gandhi Int. Airport,St. Xavier's High School,Miracles Apollo Cradle,Ambience Mall New,NH8,Hyatt Regency Gurgaon,Delhi Public School,Elan Miracle Mall,Miracles Apollo Cradle Spectra Hospital,Agri Business Management Collage,Delhi Jaipur Expressway,Grand Hyatt Gurgaon,Duke Horse Riding Club,PVR Drive In Cinema,W Pratiksha Hospital,Metro World Mall,Unicosmos School,Faridabad Gurgaon Road,Sohna Road,Bestech Business Tower,Appu Ghar,SkyJumper Trampoline Park,Axis Bank,KMP Expressway,Karma Lakelands,Jungle Safari & Trails,DPS Manesar,Medanta Hospital,Faridabad - Gurgaon Road,Lingaya's Lalita Devi Institute,ASF Insignia SEZ,Banjara Market Gurugram,Central Plaza Mall,"Paras Hospitals, Gurgaon",Badshahpur Sohna Rd Hwy,Vega School,Indian School of Hospitality,Vatika City Centre,Aatish Hospital,Info Technology Park Phase 2,Huda Metro Station,Southern Peripheral Rd,Global Ways School,Radisson Hotel,NH 248A,Sector 55/56 Metro Station,Mavens Inn,Sanar International Hospital,Sector 53-54 Metro Station,"IILM University, Gurugram",The Banyan Tree World School,The Big Tree Cafe,DLF Golf and Country Club,"Delhi Public School, Sector 84",Aarvy Hospital,DPG Degree College,Shivani public school,Baghera University,Kutumbh Hospital,Bijwasan Railway Station,Global Foyer Mall,Phase 2 Metro Station,Gurgaon Dreamz Mall,"Metro Hospital, Palam Vihar",Delhi Ajmer Expressway,Infinity Business Park,Huda metro station,Rion's Hospital,"Euro International School, Sector- 109.",Golf Course Ext Road,"Heritage Xperiential Learning, CRPF Rd",Sector 54 Chowk Metro Station

In [42]:
scaler2 = StandardScaler()

location_df_scaled = pd.DataFrame(scaler2.fit_transform(location_df), columns=location_df.columns, index=location_df.index)
location_df_scaled.head()


,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,Indira Gandhi International Airport,Hamoni Golf Camp,Fun N Food Waterpark,Accenture DDC5,DPSG Palam Vihar Gurugram,"Park Hospital, Palam Vihar",Palam Vihar Halt Railway Station,Dwarka Sector 21 Metro Station,Dwarka Expressway,Fun N Food Water Park,Tau DeviLal Sports Complex,Hyatt Place,Altrade Business Centre,AIPL Business Club Sector 62,Heritage Xperiential Learning School,CK Birla Hospital,Paras Trinity Mall Sector 63,Rapid Metro Station Sector 56,De Adventure Park,Golf Course Ext Rd,DoubleTree by Hilton Hotel Gurgaon,KIIT College of Engineering Sohna Road,Mehrauli-Gurgaon Road,Nirvana Rd,TERI Golf Course,The Shikshiyan School,WTC Plaza,Luxus Haritma Resort,BSF Golf Course,Rions Hospital,Gurgaon,Dwarka Sector 21,Nehru Stadium,Fun N Food WaterPark,IGI Airport,Vasant Kunj,Pranavananda Int. School,DLF Site central office,Holiday Inn Gurugram Sector 90,Krishna Hospital,Royal Institute Of Science,Sapphire 83 Mall,NH48,Garhi Harsaru Junction,Manesar Golf Course,AapnoGhar,Vega Schools NH-8,DLF Corporate Greens,Miracles Apollo Cradle Hospital,Hyatt Regency Gurugram,NH 48,Golden Greens Golf & Resorts Limited,Mount Olympus Junior School,Miracles Apollo Hospital,NH -8,"Savoy Suites, Manesar",Golden Greens Golf & Resorts,IMT Manesar,Amity University Gurugram,Golf Course Extension Road,"Dwarka Expy, Sector 109","Euro International School, Sector- 109",Jai Sai Ram Hospital,Aryan Hospital,Idea Cosmic Plaza,Indira Gandhi Intl Airport,Royal Institute Of Science & Management,Pataudi Road,Holiday Inn Sector 90,RPS International School,Aarvy Healthcare Hospital,Iris Broadway Mall,Imperia Mindspace,AIPL Business Tower,Heritage School,"Lotus Valley Intl School, Gurgaon",Gurugram University,Sector 55-56 Metro Station,Omaxe Gurgaon Mall,Sushant University,"Badshahpur Sohna Rd Hwy,Sector 48",Naurangpur Cricket Stadium,Naurangpur Road,National Highway 48,Vatika Town Square-INXT,Ompee International School,Manesar Bus Stand,Yashlok Medical Centre,Euro International School,WorldMark Gurgaon,Capital Cyberscape,The Shriram Millennium School,DoubleTree by Hilton Hotel,Badshahpur Sohna Hwy,Nakhrola Stadium,Delhi - Jaipur Expressway,Vatika Town Square-INXT Mall,Savoy Suites,Bal Bharati Public School,Vatika Business Centre,Indira Gandhi Int. Airport,St. Xavier's High School,Miracles Apollo Cradle,Ambience Mall New,NH8,Hyatt Regency Gurgaon,Delhi Public School,Elan Miracle Mall,Miracles Apollo Cradle Spectra Hospital,Agri Business Management Collage,Delhi Jaipur Expressway,Grand Hyatt Gurgaon,Duke Horse Riding Club,PVR Drive In Cinema,W Pratiksha Hospital,Metro World Mall,Unicosmos School,Faridabad Gurgaon Road,Sohna Road,Bestech Business Tower,Appu Ghar,SkyJumper Trampoline Park,Axis Bank,KMP Expressway,Karma Lakelands,Jungle Safari & Trails,DPS Manesar,Medanta Hospital,Faridabad - Gurgaon Road,Lingaya's Lalita Devi Institute,ASF Insignia SEZ,Banjara Market Gurugram,Central Plaza Mall,"Paras Hospitals, Gurgaon",Badshahpur Sohna Rd Hwy,Vega School,Indian School of Hospitality,Vatika City Centre,Aatish Hospital,Info Technology Park Phase 2,Huda Metro Station,Southern Peripheral Rd,Global Ways School,Radisson Hotel,NH 248A,Sector 55/56 Metro Station,Mavens Inn,Sanar International Hospital,Sector 53-54 Metro Station,"IILM University, Gurugram",The Banyan Tree World School,The Big Tree Cafe,DLF Golf and Country Club,"Delhi Public School, Sector 84",Aarvy Hospital,DPG Degree College,Shivani public school,Baghera University,Kutumbh Hospital,Bijwasan Railway Station,Global Foyer Mall,Phase 2 Metro Station,Gurgaon Dreamz Mall,"Metro Hospital, Palam Vihar",Delhi Ajmer Expressway,Infinity Business Park,Huda metro station,Rion's Hospital,"Euro International School, Sector- 109.",Golf Course Ext Road,"Heritage Xperiential Learning, CRPF Rd",Sector 54 Chowk Metro Station

In [43]:
cosine_sim3 = cosine_similarity(location_df_scaled)
cosine_sim3.shape


(246, 246)

In [45]:
def location_based_recommender(property_name, top_n=10):
    # cosine similarity matrix
    # cosine_sim_matrix = cosine_sim3
    cosine_sim_matrix = 0.5*cosine_sim + 0.8*cosine_sim2 + cosine_sim3
    
    # get the pairwise similarity scores of all properties with that property
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_scaled.index.get_loc(property_name)]))
    
    # sort the properties based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]
    top_scores = [i[1] for i in sim_scores[1:top_n+1]]
    
    top_properties = location_df_scaled.index[top_indices].tolist()
    
    # create a DataFrame with the recommendations
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'similarity_score': top_scores
    })
    
    return recommendations_df


location_based_recommender("DLF The Camellias")


,PropertyName,similarity_score
0,Salcon The Verandas,0.830510
1,DLF The Magnolias,0.607824
2,DLF The Aralias,0.551976
3,Parsvnath Exotica,0.400384
4,Pioneer Urban Presidia,0.333353
5,Tulip Monsella,0.323800
6,M3M Golfestate,0.321774
7,Pioneer Araya,0.315538
8,Mahindra Luminare,0.303470
9,Bestech Park View Grand Spa,0.296397


In [46]:
import pickle

pickle.dump(location_df, open('../Application/assets/location_dist.pkl', 'wb'))


In [47]:
pickle.dump(cosine_sim, open('../Application/assets/cosine_sim.pkl', 'wb'))
pickle.dump(cosine_sim2, open('../Application/assets/cosine_sim2.pkl', 'wb'))
pickle.dump(cosine_sim3, open('../Application/assets/cosine_sim3.pkl', 'wb'))
